## Problem: 3Sum

Given an integer array `nums`, return all the triplets `[nums[i], nums[j], nums[k]]` such that `i != j`, `i != k`, and `j != k`, and `nums[i] + nums[j] + nums[k] == 0`.

Notice that the solution set must not contain duplicate triplets.

### Example 1:
```
Input: nums = [-1,0,1,2,-1,-4]
Output: [[-1,-1,2],[-1,0,1]]
Explanation:
nums[0] + nums[1] + nums[2] = (-1) + 0 + 1 = 0.
nums[1] + nums[2] + nums[4] = 0 + 1 + (-1) = 0.
nums[0] + nums[3] + nums[4] = (-1) + 2 + (-1) = 0.
The distinct triplets are [-1,0,1] and [-1,-1,2].
Notice that the order of the output and the order of the triplets does not matter.
```

### Example 2:
```
Input: nums = [0,1,1]
Output: []
Explanation: The only possible triplet does not sum up to 0.
```

### Example 3:
```
Input: nums = [0,0,0]
Output: [[0,0,0]]
Explanation: The only possible triplet sums up to 0.
```

### Constraints:
- `3 <= nums.length <= 3000`
- `-10^5 <= nums[i] <= 10^5`

```python
def three_sum(nums):
    pass
```

## 1st Try: imitatation for 2 sum

In [25]:
def three_sum_v1(nums):
    nums.sort()
    length = len(nums)
    ans = list()
    for i in range(length):
        target = -1 * nums[i]
        left = i + 1
        right = length - 1
        while left < right:
            if nums[left] + nums[right] < target:
                left += 1
            elif nums[left] + nums[right] > target:
                right -= 1
            else:
                ans.append([-1 * target, nums[left], nums[right]])
                left += 1
                right -= 1
    return ans

In [26]:
# Input: nums = [-1,0,1,2,-1,-4]
# Output: [[-1,-1,2],[-1,0,1]]

nums = [-1, 0, 1, 2, -1, -4]
three_sum_v1(nums)

[[-1, -1, 2], [-1, 0, 1], [-1, 0, 1]]


### ❌ Bug 1: No duplicate skipping for `i`
For `[-1, -1, 0, 1, 2]`, you'll process `i=0` (num=-1) and `i=1` (num=-1) separately, producing the same triplets twice.

**Fix:** Add at the start of the for loop:
```python
if i > 0 and nums[i] == nums[i-1]:
    continue

In [27]:
## 2nd Try

In [45]:
def three_sum_v2(nums):
    nums.sort()
    length = len(nums)
    ans = list()
    for i in range(length):
        if i > 0 and nums[i] != nums[i - 1]:
            target = -1 * nums[i]
            left = i + 1
            right = length - 1
            while left < right:
                while nums[left] + nums[right] < target:
                    left += 1
                while nums[left] + nums[right] > target:
                    right -= 1
                if nums[left] + nums[right] == 0:
                    ans.append([-1 * target, nums[left], nums[right]])
                left += 1
                right -= 1
    return ans


In [46]:
nums = [-1, 0, 1, 2, -1, -4]
three_sum_v2(nums)

IndexError: list index out of range

### ❌ Bug 1: Duplicate skip logic is inverted
```python
if i > 0 and nums[i] != nums[i - 1]:  # ❌ processes only when DIFFERENT from previous
```
You're skipping the first occurrence and processing duplicates. You want the opposite.

**Fix:**
```python
if i > 0 and nums[i] == nums[i - 1]:
    continue  # skip duplicates, keep the first
```

---

### ❌ Bug 2: Inner `while` loops can go out of bounds
```python
while nums[left] + nums[right] < target:
    left += 1
```
If no pair meets the condition, `left` crosses `right` → `IndexError`.

**Fix:** Always include `left < right`:
```python
while left < right and nums[left] + nums[right] < target:
    left += 1
while left < right and nums[left] + nums[right] > target:
    right -= 1
```

---

### ❌ Bug 3: Wrong equality check
```python
if nums[left] + nums[right] == 0:  # ❌ should compare to target
```
**Fix:**
```python
if nums[left] + nums[right] == target:
```

---

### ❌ Bug 4: Moves pointers even without a match
```python
left += 1
right -= 1
```
These run unconditionally at the end of the while loop. If the inner whiles broke because `left >= right`, you'll skip valid pairs. Only move pointers **after finding a match**.

**Fix:** Use `if/elif/else` structure like your v1:
```python
if nums[left] + nums[right] < target:
    left += 1
elif nums[left] + nums[right] > target:
    right -= 1
else:
    # match found
    ans.append(...)
    left += 1
    right -= 1
```

---

### ❌ Bug 5: No duplicate skipping for `left` after a match
For input like `[-2, 0, 0, 2, 2]`, you'll add `[-2, 0, 2]` multiple times.

**Fix:** After a match, skip same values:
```python
while left < right and nums[left] == nums[left - 1]:
    left += 1
```

---

### 💡 Summary
Go back to your v1 structure (`if/elif/else`) and add just two things:
1. Skip duplicate `i`: `if i > 0 and nums[i] == nums[i-1]: continue`
2. Skip duplicate `left` after match: `while left < right and nums[left] == nums[left-1]: left += 1`

In [75]:
def three_sum_v3(nums):
    nums.sort()
    length = len(nums)
    ans = list()
    for i in range(length):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        target = -1 * nums[i]
        left = i + 1
        right = length - 1
        while left < right:
            if nums[left] + nums[right] < target:
                left += 1
            elif nums[left] + nums[right] > target:
                right -= 1
            else:
                ans.append([-1 * target, nums[left], nums[right]])
                left += 1
                right -= 1
    return ans

In [76]:
nums = [-1, 0, 1, 2, -1, -4]
three_sum_v3(nums)

[[-1, -1, 2], [-1, 0, 1]]

In [77]:
nums = [-2, 0, 0, 2, 2]
three_sum_v3(nums)

[[-2, 0, 2]]

### ❌ Problem
For `[-2, 0, 0, 2, 2]`, after finding `[-2, 0, 2]`:
- `left` moves from index 1 (value 0) to index 2 (value 0 again)
- The next iteration finds the same triplet `[-2, 0, 2]` → duplicate


## 4th Try

In [7]:
def three_sum_v4(nums):
    nums.sort()
    length = len(nums)
    ans = list()
    for i in range(length):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        target = -1 * nums[i]
        left = i + 1
        right = length - 1
        while left < right:
            if nums[left] + nums[right] < target:
                left += 1
            elif nums[left] + nums[right] > target:
                right -= 1
            else:
                ans.append([-1 * target, nums[left], nums[right]])
                left += 1
                right -= 1
                while left < right and nums[left] == nums[left - 1]:
                    left += 1
    return ans

In [8]:
nums = [-1, 0, 1, 2, -1, -4]
three_sum_v4(nums)

[[-1, -1, 2], [-1, 0, 1]]

In [9]:
nums = [-2, 0, 0, 2, 2]
three_sum_v4(nums)

[[-2, 0, 2]]